In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA causal_project;

In [0]:
%sql
SELECT
  treatment,
  COUNT(*)                        AS n,
  AVG(pre_avg_weekly_spend)       AS avg_weekly_spend,
  AVG(pre_coupon_usage_rate)      AS coupon_rate,
  AVG(pre_loyalty_card_rate)      AS loyalty_rate,
  AVG(pre_avg_weekly_trips)       AS weekly_trips,
  AVG(pre_department_breadth)     AS dept_breadth,
  AVG(avg_daily_campaign_spend)   AS outcome,
  AVG(clean_window_days) AS campaign_days,
  AVG(pre_active_weeks) AS pre_active_weeks
FROM causal_project.gold_household
GROUP BY treatment;

In [0]:
gold_df = spark.table("gold_household").toPandas()

In [0]:
gold_df.head()


In [0]:
from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
import pandas as pd

continuous_cols = ["pre_avg_weekly_spend","pre_avg_weekly_trips","pre_coupon_usage_rate", "pre_loyalty_card_rate", "pre_department_breadth","pre_active_weeks"]

cat_cols = ['age_group','marital_status','income_level',
            'household_size','home_ownership','kid_count']

for col in continuous_cols:

    t,p = ttest_ind(

        gold_df[gold_df.treatment==1][col],

        gold_df[gold_df.treatment==0][col],

        equal_var=False

    )

    print(col, p)
results = []

for col in cat_cols:

    contingency = pd.crosstab(

        gold_df[col],

        gold_df["treatment"]

    )

    chi2, p, dof, expected = chi2_contingency(contingency)

    results.append({

        "variable": col,

        "chi2": chi2,

        "dof": dof,

        "p_value": p

    })

chi_results = pd.DataFrame(results).sort_values("p_value")

display(chi_results)